# Kiểm tra tài khoản trong kernel Colab
Chọn Colab T4 hoặc mở trực tiếp trên web Colab. Cell từ chối kernel local để tránh đọc nhầm GPU/ổ máy cá nhân thành quota Colab. Tài khoản Drive API có thể khác tài khoản cấp GPU.


In [ ]:
# Paste this entire file into ONE Colab cell; no project imports needed.
from pathlib import Path
import shutil
import subprocess


def check_colab_quota():
    # A local GPU/disk is not evidence about the user's Colab account.
    try:
        from google.colab import auth
    except ImportError:
        print('CHƯA KIỂM TRA TÀI KHOẢN: cell đang chạy ở kernel local, không phải Colab.')
        print('Trong VS Code: Select Kernel → Colab → chọn/kết nối server mới.')
        print('Hoặc mở notebook này trực tiếp tại https://colab.research.google.com rồi chạy.')
        print('Không kết luận được quota GPU hoặc dung lượng Drive từ kernel local.')
        return

    def gib(value):
        return f'{int(value) / 2**30:.2f} GiB'

    print('=== GPU được cấp cho runtime hiện tại ===')
    try:
        gpu = subprocess.run(
            ['nvidia-smi', '--query-gpu=name,memory.total,memory.used', '--format=csv,noheader'],
            capture_output=True, text=True, timeout=15)
        if gpu.returncode == 0 and gpu.stdout.strip():
            print('Tên GPU, VRAM tổng (MiB), VRAM dùng (MiB):')
            print(gpu.stdout.strip())
        else:
            print('Không đọc được GPU. Có thể là runtime CPU, lỗi driver hoặc chưa được cấp GPU.')
    except (OSError, subprocess.TimeoutExpired):
        print('Không đọc được GPU. Không thể suy ra quota còn lại từ việc này.')

    disk = shutil.disk_usage('/content' if Path('/content').is_dir() else '.')
    print('\nỔ tạm runtime còn:', gib(disk.free), '/', gib(disk.total), '(khác Google Drive)')
    print('Drive đã mount:', Path('/content/drive/MyDrive').is_dir())
    print('\n=== Tài khoản và dung lượng qua Drive API ===')
    print('Chọn tài khoản Google khi được hỏi. Không in hoặc lưu token.')
    try:
        import google.auth
        from google.auth.transport.requests import AuthorizedSession
        auth.authenticate_user()
        credentials, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive.metadata.readonly'])
        with AuthorizedSession(credentials) as session:
            response = session.get('https://www.googleapis.com/drive/v3/about',
                                   params={'fields': 'user(displayName,emailAddress),storageQuota'}, timeout=30)
        if response.ok:
            info = response.json()
            user, quota = info.get('user', {}), info.get('storageQuota', {})
            print('Tài khoản Drive API:', user.get('emailAddress', '(không có email)'),
                  '|', user.get('displayName', ''))
            print('Tài khoản này có thể khác tài khoản đang dùng Colab hoặc mount Drive; đối chiếu avatar Google.')
            used = quota.get('usage')
            limit = quota.get('limit')
            print('Đã dùng trên các dịch vụ Google:', gib(used) if used is not None else 'API không trả số liệu')
            if used is not None and limit is not None:
                free = max(0, int(limit) - int(used))
                print('Giới hạn lưu trữ:', gib(limit), '| Còn trống:', gib(free))
                if free == 0:
                    print('Dung lượng lưu trữ đã hết; có thể không ghi được features/checkpoint.')
            else:
                print('Không tính được dung lượng còn lại: API không trả đủ limit/usage.')
            print('Riêng Drive:', gib(quota['usageInDrive']) if 'usageInDrive' in quota else 'không có số liệu')
            print('Trong thùng rác:', gib(quota['usageInDriveTrash']) if 'usageInDriveTrash' in quota else 'không có số liệu')
            print('Workspace dùng pooled storage: limit/usage có thể là của tổ chức.')
        else:
            print('Không đọc được Drive quota. HTTP:', response.status_code,
                  '— kiểm tra đăng nhập, quyền Drive API; không suy ra là hết quota.')
    except Exception as error:
        print('Không hoàn tất kiểm tra Drive:', type(error).__name__)

    print('\n=== Giới hạn GPU Colab ===')
    print('Không xác định được số giờ GPU miễn phí còn lại hoặc thời điểm reset bằng cell này.')
    print('Colab không công bố giới hạn cố định; có GPU hiện tại cũng không chứng minh còn bao nhiêu quota.')
    print('Nếu không kết nối được GPU: có thể chọn runtime CPU để chạy riêng cell kiểm tra Drive.')
    print('Đối chiếu đúng lỗi: GPU usage limits / Drive storage quota / Drive download hoặc I/O quota là các loại khác nhau.')
    print('Xem dung lượng Google: https://one.google.com/storage')
    print('Giới hạn Colab: https://research.google.com/colaboratory/faq.html')


check_colab_quota()
